# Naturalistic EMA prediction based on ACC - Dyskinesia use case

## 0) Imports

- document versions for reproducibility

In [ ]:
# import packages
import datetime as dt
import pandas as pd
import numpy as np
import os
import sys
import csv
import json
import importlib
from itertools import product, compress
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, variation

from dataclasses import dataclass, field


In [ ]:
print('Python sys', sys.version)
print('pandas', pd.__version__)
print('numpy', np.__version__)
# print('mne_bids', mne_bids.__version__)
# print('mne', mne.__version__)
# print('sci-py', scipy.__version__)
# print('sci-kit learn', sk.__version__)
# print('matplotlib', plt_version)

"""
Python sys 3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)]
pandas 2.1.1
numpy 1.26.0

from 16.09

Python sys 3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)]
pandas 2.3.2
numpy 2.3.3
"""

Import custom functions

In [ ]:
# from dbs_home repo
from dbs_home.load_raw.main_load_raw import loadSubject 
import dbs_home.utils.helpers as home_helpers
import dbs_home.utils.ema_utils as home_ema_utils
import dbs_home.plot_data.plot_compliance as plot_home_compl
import dbs_home.preprocessing.preparing_ema as home_ema_prep

from dbs_home.preprocessing import get_submovements
import dbs_home.preprocessing.acc_preprocessing as acc_prep
import dbs_home.preprocessing.submovement_processing as submove_proc
import dbs_home.load_raw.load_watch_raw as load_watch


In [ ]:
# from current repo
from utils import load_utils, load_data, prep_data
import utils.acc_features as acc_fts
import utils.ft_extract_class as ft_extr_class
import utils.feat_extraction as ft_extr
import utils.data_handling_ema_acc as data_handling

from plotting import plot_help



## 1) Load and check naturalistic data

In [ ]:
# set paths

feas_data_path = os.path.join(
    os.path.dirname(load_utils.get_onedrive_path()),
    'PROJECTS', 'home_feasibility'
)
feas_fig_path = os.path.join(
    load_utils.get_onedrive_path('figures'),
    'feasibility'
)



#### Load ACC data
- create SVM
- filter data within the dataclass

In [ ]:
# LID
sub_id = 'hm24'
ses_id = 'ses03'

FT_PARAMS_VERSION = 'v3'  # v3 updated lid version

In [ ]:
# import naturalistic data via dbs_home repo




### test days for hm24-ses01  # dyskinesia
# dev_day_selection = ['2025-07-17', '2025-07-18']
# dev_day_selection = [f'2025-07-{d}' for d in np.arange(17, 31)]
dev_day_selection = []

### test days for hm20-ses01  # tremor
# dev_day_selection = [
#     '2025-06-13', '2025-06-14',
#     '2025-06-15', '2025-06-16'
# ]

home_dat = loadSubject(
    sub=sub_id,
    ses=ses_id,
    incl_STEPS=False,
    incl_EPHYS=False,
    incl_EMA=True,
    incl_ACC=True,
    day_selection=dev_day_selection
)

Check available EMAs

In [ ]:
plot_home_compl.plot_EMA_completion_perSession(home_dat)

## 2) ACC processing incl. Feature extraction

#### Extract features from Acc-Windows aligned to EMAs

full day extraction and visualization at end of notebook

In [ ]:
importlib.reload(ft_extr_class)
importlib.reload(acc_fts)
importlib.reload(data_handling)
importlib.reload(ft_extr)



# iter_settings = {
#     'nosm_allwindow':[False, True, True],
#     'sm_merged': [True, False, False],
#     'sm_single': [True, False, True]}

ft_settings = {
    'nosm_allwindow': {
        'EXTRACT_FT_FROM_SMs': False,
        'EXTRACT_FT_FULL_WIN': True,
        'ACC_FEATS_on_SINGLE_MOVES': True
    },
    'sm_merged': {
        'EXTRACT_FT_FROM_SMs': True,
        'EXTRACT_FT_FULL_WIN': False,
        'ACC_FEATS_on_SINGLE_MOVES': False
    },
    'sm_single': {
        'EXTRACT_FT_FROM_SMs': True,
        'EXTRACT_FT_FULL_WIN': False,
        'ACC_FEATS_on_SINGLE_MOVES': True
    }
}



fullday_dict = {}

# for ft_type, ft_settings in ft_settings.items():

ft_settings = ft_settings['sm_merged']  # use sm_merged settings for all sessions for now
    
for ses_id in ['ses01', 'ses02', 'ses03']:

    # fullday_dict[ses_id] = ft_extr.get_features_per_session(
    #     home_dat=home_dat,
    #     sub_id=sub_id,
    #     ses_id=ses_id,
    #     FT_PARAMS_VERSION=FT_PARAMS_VERSION,
    #     LOAD_SAVE_FEATS = True,
    #     # define how features should be extracted
    #     STORE_SUBMOVES = False,
    #     # plotting settings
    #     SAVE_PLOT = False,
    #     SHOW_PLOT = False,
    #     **ft_settings
    # )
    
    fullday_dict[ses_id] = ft_extr.get_feat_df_for_pred(
        sub_id=sub_id,
        ses_id=ses_id,
        ft_set_sel='sm_merged',
        FT_PARAMS_VERSION=FT_PARAMS_VERSION,
        ONLY_EMA_WINDOWS=False,
    )


In [ ]:
[f'{k}: {v.shape}, column-names: {list(v.keys())}' for k, v in fullday_dict.items()]

## 3a) Evaluate extracted Features

#### TODO: include z-scoring outlier removal


In [ ]:
from plotting import plot_home_preds as plt_preds
import plotting.plot_ft_explor as plt_fts

define selected data

In [ ]:
sub_id = 'hm24'
SES = 'ses01'
FT_TYPE = 'sm_merged'    # nosm_allwindow, sm_merged, sm_single
FT_PARAMS_VERSION = 'v3'  # v3 updated lid version
EMA_ref = 'LID'



In [ ]:
importlib.reload(ft_extr)


xy_dict = {}

for ses_id in ['ses01', 'ses03']:   #'ses02', 
    
    tempdf = ft_extr.get_feat_df_for_pred(
        sub_id=sub_id,
        ses_id=ses_id,
        ft_set_sel=FT_TYPE,
        FT_PARAMS_VERSION=FT_PARAMS_VERSION,
        ONLY_EMA_WINDOWS=True,
    )
    xy_dict[ses_id] = tempdf

In [ ]:
ft_df = xy_dict['ses01'].copy()


check selected variables in dataframe and first rows of dataframe

In [ ]:
print(ft_df.shape)
print(ft_df.keys())
print(ft_df.head())
print([f'LID "{k}": {v} counts' for k, v in zip(
    *np.unique(ft_df[EMA_ref], return_counts=True))])


In [ ]:
importlib.reload(ft_extr)

# define which features to include in prediction and which to exclude (times, ema)
feats_incl, EMA_CODING = ft_extr.get_feat_params(FT_PARAMS_VERSION)
keys_excl = ['timestamp'] + list(EMA_CODING.keys())  # times + ema

Visualize feature distributions

In [ ]:
importlib.reload(plt_fts)

save_plot = False


for ft_name in ft_df.keys():
    print(f'\n{ft_name}')
    # if ft_name not in ft_df.keys():
    #     print(f'Feature {ft_name} not found in dataframe columns. Skipping.')
    #     continue
    if ft_name in keys_excl:
        print(f'Feature {ft_name} is not included in the feature set. Skipping.')
        continue
    
    ### plot distributions with histo, scatter, and boxes
    plt_fts.plot_ft_distribution(
        ft_df=ft_df,
        ft_name=ft_name,
        EMA_ref=EMA_ref,
        sub_id=sub_id,
        SES=SES,
        FT_TYPE=FT_TYPE,
        FT_PARAMS_VERSION=FT_PARAMS_VERSION,
        save_plot=save_plot,
    )



### Test: feature distribution after PCA and taking first X components

In [ ]:
from sklearn.decomposition import PCA
import utils.pred_utils as pred_utils
import utils.pred_clustering as clust


In [ ]:
EMA_Y = 'LID'
EXCL_HR = False  # exclude heart rate features (not available in all timepoints)
SKEW_THRESH = 1  # None is no log-transform, otherwise log-transform features with skew above threshold
OUTLIER_REMOVAL = 'zscore'  # 'none', 'zscore', 'isoforest'
USE_PCA=True
# N_CLUST = 3

TRANSFORM_Y = None
# TRANSFORM_Y = {1: [1,], 2: [2, 3, 4, 5,], 3: [6, 7, 8, 9]}  # transform LID into 3 classes: 1, 2 (1-4), 3 (>4)



run functions from run_clustering()

In [ ]:

# define window for clustering
cv_df = xy_dict['ses01'].copy()
cv_df = pd.concat([cv_df, xy_dict['ses03']]).reset_index(drop=True) 


PRED_FTS = pred_utils.get_keys_incl(cv_df.keys(), excl_hr=EXCL_HR,)
print(f'model trained on features: {PRED_FTS}')




In [ ]:
### define X, y
X = cv_df[PRED_FTS].values.copy()
y = cv_df[EMA_Y].values.copy().astype(float)

if TRANSFORM_Y is not None:
    for new_y, old_y_list in TRANSFORM_Y.items():
            y[np.isin(y, old_y_list)] = new_y

# returns z-scored features
X, y = clust.prepare_X_for_clustering(
    X=X, y=y, skew_threshold=SKEW_THRESH, verbose=True,
)
print('all data:', X.shape, y.shape)
if OUTLIER_REMOVAL == 'isoforest':
    X, y = clust.remove_outliers_isoforest(
        X, y=y, verbose=True,
    )

elif OUTLIER_REMOVAL == 'zscore':
    X, y = clust.remove_outliers_zscore(
        X=X, y=y, threshold_n_sd=3.0, verbose=True,
    )
print('after outlier removal:', X.shape, y.shape)

X = PCA(n_components=6).fit_transform(X)

print('post-PCA:', X.shape, y.shape)


In [ ]:
### visualize created PCAs

# plot PC-1 and -2, color-code for tremor score
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for i_ax, pc2 in enumerate([1, 2]):
    ax = axes[i_ax]
    im = ax.scatter(X[:, 0], X[:, pc2], c=y, cmap='viridis', edgecolor='k')
    ax.set_xlabel('Principal Component 1')
    ax.set_ylabel(f'Principal Component {pc2+1}')
    ax.set_title(f'PC-1 vs PC-{pc2+1}')
fig.colorbar(im, label='Dyskinesia score',)
plt.tight_layout()
plt.show()

for i_pc in np.arange(X.shape[1]):
    fig, ax = plt.subplots(1, 1)

    im = ax.scatter(y, X[:, i_pc], c=y, cmap='viridis', edgecolor='k')

    ax.set_xlabel('LID (EMA score)')
    ax.set_ylabel(f'PC # {i_pc+1} (a.u.)')

    plt.show()

## 3b) Predict EMAs based on Wearable-features

#### LID-classification: cross-validation based on preop session

- only EMA windows
- cv-folds: n-folds of EMA-windows from all sessions

In [ ]:

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge, LogisticRegression

from sklearn.metrics import (
    r2_score, root_mean_squared_error,
    mean_absolute_error, cohen_kappa_score
)
from scipy.stats import spearmanr, kendalltau, mannwhitneyu


get X and y data, preprocess data

In [ ]:
# define window for cross-validation
cv_df = xy_dict['ses01'].copy()
cv_df = pd.concat([cv_df, xy_dict['ses03']]).reset_index(drop=True) 

print(cv_df.keys())

FT_TYPE = 'sm_merged'    # nosm_allwindow, sm_merged, sm_single
FT_PARAMS_VERSION = 'v3'  # v3 updated lid version
EMA_Y = 'LID'
# TRANSFORM_Y = {1: [1,], 2: [2, 3, 4, 5,], 3: [6, 7, 8, 9]}  # transform LID into 3 classes: 1, 2 (1-4), 3 (>4)
TRANSFORM_Y = None

EXCL_HR = True  # exclude heart rate features (not available in all timepoints)
LOG_SKEWED = True
Z_STD_OUTLIER_THRESH = 3.0
APPLY_PCA = True
N_PCA_COMP=6

if TRANSFORM_Y is not None:
    REGR_MODEL = LogisticRegression(multi_class='multinomial',)
else:
    REGR_MODEL = Ridge(alpha=1.0)

# PRED_FTS = pred_utils.get_keys_incl(cv_df.keys(), excl_hr=EXCL_HR,)

# print(f'model trained on features: {PRED_FTS}')

In [ ]:
importlib.reload(pred_utils)

output_cv = pred_utils.full_preproc_X_y_regr(
    df=cv_df,
    EMA_Y=EMA_Y,
    EXCL_HR=EXCL_HR,
    LOG_SKEWED=LOG_SKEWED,
    TRANSFORM_Y=TRANSFORM_Y,
    Z_STD_OUTLIER_THRESH=Z_STD_OUTLIER_THRESH,
    APPLY_PCA=APPLY_PCA,
    PCA_n_comps=N_PCA_COMP,
)
X_all = output_cv['X_all']
y_all = output_cv['y_all']

In [ ]:
importlib.reload(pred_utils)


print(F'regression model selected: {REGR_MODEL}')
# pipe includes scaling based on training data, and regression model
cv_pipe = Pipeline([
    # ('corr_filter', pred_utils.DropCorrelatedFeatures(threshold=0.9)),
    # ('scaler', StandardScaler()),
    ('reg', REGR_MODEL),
])

y_pred_all = np.zeros_like(y_all)

n_folds = 4
cv_folds = KFold(n_splits=n_folds).split(X_all)

for i_fold, (train_idx, test_idx) in enumerate(cv_folds):
    print(f'test indices for fold {i_fold + 1}: {test_idx}')
    X_train, X_test = X_all[train_idx], X_all[test_idx]
    y_train, y_test = y_all[train_idx], y_all[test_idx]

    # train regression model
    cv_pipe.fit(X_train, y_train)
    # predict test set (incl scaling of fts based on train set)
    y_pred = cv_pipe.predict(X_test)
    # clip predictions to valid range and round to nearest integer
    y_pred = np.clip(np.round(y_pred), y_train.min(), y_train.max()).astype(int)
    y_pred_all[test_idx] = y_pred  # add predictions to total array in correct indices
    
    print(f'finished fold {i_fold + 1} / {n_folds}')

# autocorr_col_idx = [1, 4, 5, 6, 11, 14, 19, 20, 21, 25]
# print(f'feats often removed: {np.array(PRED_FTS)[autocorr_col_idx]}')


In [ ]:
# if (FT_TYPE == 'sm_merged' and FT_PARAMS_VERSION == 'v3' and EXCL_HR == False
#     and LOG_SKEWED == True):
#     autocorr_drop_col_idx = [1, 4, 5, 6, 11, 14, 19, 20, 21, 25]
# elif (FT_TYPE == 'sm_merged' and FT_PARAMS_VERSION == 'v3' and EXCL_HR == False
#     and LOG_SKEWED == False):
#     autocorr_drop_col_idx = [1, 4, 5, 6, 11, 14, 19, 21, 28]

# print(f'selected drop col idx: {autocorr_drop_col_idx}')

quick evaluation, check sign with permutations

In [ ]:
rho, p = spearmanr(y_all, y_pred_all)
print(f"Spearman rho={rho.round(2)}, p={p.round(5)}")

tau, p_tau = kendalltau(y_all, y_pred_all)
print(f"Kendall tau={tau.round(2)}, p={p_tau.round(5)}")  

mae = mean_absolute_error(y_all, y_pred_all)
kappa = cohen_kappa_score(y_all, y_pred_all, weights='linear')
print(f"Mean Absolute Error: {mae:.2f}, Cohen's Kappa: {kappa:.2f}")

r2 = r2_score(y_all, y_pred_all)
rmse = root_mean_squared_error(y_all, y_pred_all)
print(f"### R2 for all data: {r2:.2f}, RMSE: {rmse:.2f}")

cv_results = {
    'spearman_rho': rho,
    'spearman_p': p,
    'kendall_tau': tau,
    'kendall_p': p_tau,
    'mae': mae,
    'kappa': kappa,
    'r2': r2,
    'rmse': rmse
}

fig, ax = plt.subplots(1, 1, figsize=(6, 6))

y_jitter = np.random.normal(0, 0.1, size=y_all.shape)  # add jitter for better visibility
x_jitter = np.random.normal(0, 0.1, size=y_all.shape)  # add jitter for better visibility
ax.scatter(y_all + x_jitter, y_pred_all + y_jitter, alpha=0.5)
ax.set_xlabel('True LID')
ax.set_ylabel('Predicted LID')

plt.show()

## 3c) Test LID classifier on post-op sessions

- Train-split: EMA-windows ses01-03, Test-split: all-windows ses01 and ses03

In [ ]:
### DEFINE PARAMETERS

FT_PARAMS_VERSION = 'v3'  # v3 updated lid version
FT_TYPE = 'sm_single'    # nosm_allwindow, sm_merged, sm_single
TEST_EMA_WIN_ONLY = False

xy_dict = {}
for ses_id in ['ses01', 'ses03']:   #'ses02', 
    tempdf = ft_extr.get_feat_df_for_pred(
        sub_id=sub_id,
        ses_id=ses_id,
        ft_set_sel=FT_TYPE,
        FT_PARAMS_VERSION=FT_PARAMS_VERSION,
        ONLY_EMA_WINDOWS=True,
    )
    xy_dict[ses_id] = tempdf

# define train data (ema-only ses01+03)
train_df = xy_dict['ses01'].copy()
train_df = pd.concat([train_df, xy_dict['ses03']]).reset_index(drop=True) 

print(train_df.keys())

EMA_Y = 'LID'
EXCL_HR = True  # exclude heart rate features (not available in all timepoints)
LOG_SKEWED = True
Z_STD_OUTLIER_THRESH = 3.0
# TRANSFORM_Y = {1: [1,], 2: [2, 3, 4, 5,], 3: [6, 7, 8, 9]}  # transform LID into 3 classes: 1, 2 (1-4), 3 (>4)
TRANSFORM_Y = None

if TRANSFORM_Y is not None:
    REGR_MODEL = LogisticRegression(multi_class='multinomial',)
else:
    REGR_MODEL = Ridge(alpha=1.0)


In [ ]:
importlib.reload(pred_utils)

train_output = pred_utils.full_preproc_X_y_regr(
    df=train_df,
    EMA_Y=EMA_Y,
    EXCL_HR=EXCL_HR,
    LOG_SKEWED=LOG_SKEWED,
    TRANSFORM_Y=TRANSFORM_Y,
    Z_STD_OUTLIER_THRESH=Z_STD_OUTLIER_THRESH,
    APPLY_PCA=APPLY_PCA,
    PCA_n_comps=N_PCA_COMP,
    return_trained_pca=True,
    return_skew_feat_bool=True,
)
X_train = train_output['X_all']
y_train = train_output['y_all']
train_skewed_ft_bool = train_output['skew_list']
trained_pca = train_output['pca']

test_pipe = Pipeline([
    # ('corr_filter', pred_utils.DropCorrelatedFeatures(threshold=0.9)),
    # ('scaler', StandardScaler()),
    ('reg', REGR_MODEL),
])


# train regression model
test_pipe.fit(X_train, y_train)

In [ ]:
### EXTRACT TEST DATA

# TODO: RUN SM_SINGLE FEATURES FOR ALL DAY SESSIONS LID

test_sess_ids = []  # store to which session test-samples belong

for i, ses_id in enumerate(['ses01', 'ses02', 'ses03']):
    
    tempdf = ft_extr.get_feat_df_for_pred(
        sub_id=sub_id,
        ses_id=ses_id,
        ft_set_sel=FT_TYPE,
        FT_PARAMS_VERSION=FT_PARAMS_VERSION,
        ONLY_EMA_WINDOWS=TEST_EMA_WIN_ONLY,
    )
    if i == 0: test_df = tempdf.copy()
    else: test_df = pd.concat([test_df, tempdf],).reset_index(drop=True)
    test_sess_ids.extend([ses_id] * len(tempdf))

print(f'test_df shape: {test_df.shape}\n\ntestdf columns: {test_df.keys()}')

In [ ]:
importlib.reload(pred_utils)

output_test = pred_utils.full_preproc_X_y_regr(
    df=test_df,
    EMA_Y=None,
    EXCL_HR=EXCL_HR,
    LOG_SKEWED=LOG_SKEWED,
    TRANSFORM_Y=TRANSFORM_Y,
    Z_STD_OUTLIER_THRESH=Z_STD_OUTLIER_THRESH,
    APPLY_PCA=APPLY_PCA,
    apply_trained_pca=trained_pca,
    PCA_n_comps=N_PCA_COMP,
    use_skew_feat_bool=train_skewed_ft_bool,
    adjust_session_list=test_sess_ids,
)
X_test = output_test['X_all']
test_sessions = output_test['session_ids']

# predict test set (incl scaling of fts based on train set)
y_test_pred = test_pipe.predict(X_test)
# clip predictions to valid range and round to nearest integer
y_test_pred = np.clip(np.round(y_test_pred), 1, 9).astype(int)


#### Evaluate full-day test predictions
- overall values preop vs postop



In [ ]:
from plotting import plot_home_preds as plt_preds

In [ ]:
SES_COLORS = {'ses01': 'violet', 'ses02': 'orange', 'ses03': 'darkcyan'}
SES_LABELS = {'ses01': 'Pre-DBS',
              'ses02': 'Post-DBS',
              'ses03': 'Post-DBS\n(optimized)'}
FS = 14

In [ ]:
### perform LMM for test evaluation

# per combi of sessions separate, ALPHA = 0.05 / 3
# get unique ids in test data
avail_sess = np.unique(test_sessions)

for ses_A, ses_B in product(avail_sess, repeat=2):
    if ses_A >= ses_B:  # only compare each pair once and skip self-comparison
        continue
    preds_A = y_test_pred[test_sessions == ses_A]
    preds_B = y_test_pred[test_sessions == ses_B]

    # depend_var = np.concat([preds_A, preds_B])
    # indep_var = ['A'] * len(preds_A) + ['B'] * len(preds_B)
    # random_var = DAY_LIST



In [ ]:
# get unique ids in test data
avail_sess = np.unique(test_sessions)

for ses_A, ses_B in product(avail_sess, repeat=2):
    if ses_A >= ses_B:  # only compare each pair once and skip self-comparison
        continue
    group_A = y_test_pred[test_sessions == ses_A]
    group_B = y_test_pred[test_sessions == ses_B]
    
    # perform Mann-Whitney U test
    stat, p_value = mannwhitneyu(group_A, group_B, alternative='two-sided')
    
    print(f'Mann-Whitney U test between {ses_A} and {ses_B}: U={stat}, p={p_value:.4f}')

## consider to premute the U-values for significance

In [ ]:
### plot predicted LID distributions per session

fig, ax = plt.subplots(1, 1, figsize=(4, 6))

ses_groups = [y_test_pred[test_sessions == ses_id] for ses_id in avail_sess]

bp = ax.boxplot(ses_groups,
    labels=[SES_LABELS[s] for s in avail_sess],
    patch_artist=True,
    medianprops=dict(color='k')
)
for box, ses in zip(bp['boxes'], avail_sess):
    box.set_facecolor(SES_COLORS[ses])

ax.tick_params(axis='both', size=FS, labelsize=FS,)
ax.set_ylabel('Predicted Dyskinesia (EMA score)', size=FS,)
ax.set_ylim(1, 9)
plt.tight_layout()
plt.show()


#### 3d) Test-validation for EMA-only predictions

-> not prefered due to test data currently used for training

In [ ]:
rho, p = spearmanr(y_test, y_pred)
print(f"Spearman rho={rho:.2f}, p={p:.5f}")

tau, p_tau = kendalltau(y_test, y_pred)
print(f"Kendall tau={tau:.2f}, p={p_tau:.5f}")  

mae = mean_absolute_error(y_test, y_pred)
kappa = cohen_kappa_score(y_test, y_pred, weights='linear')
print(f"Mean Absolute Error: {mae:.2f}, Cohen's Kappa: {kappa:.2f}")

r2 = r2_score(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
print(f"### R2 for all data: {r2:.2f}, RMSE: {rmse:.2f}")


test_results = {
    'spearman_rho': rho,
    'spearman_p': p,
    'kendall_tau': tau,
    'kendall_p': p_tau,
    'mae': mae,
    'kappa': kappa,
    'r2': r2,
    'rmse': rmse
}


fig, ax = plt.subplots(1, 1, figsize=(6, 6))

# y_jitter = np.random.normal(0, 0.1, size=y_test.shape)  # add jitter for better visibility
# ax.scatter(y_test + y_jitter, y_pred + y_jitter, alpha=0.5)

# boxplots of predicted value per true LID class
data_to_plot = [y_pred[y_test == lid] for lid in np.unique(y_test)]
ax.boxplot(data_to_plot, labels=np.unique(y_test))

ax.set_xlabel('True LID')
ax.set_ylabel('Predicted LID')

plt.show()

try permutations for EMA window data

In [ ]:
perm_pipe = Pipeline([
    # ('scaler', StandardScaler()),
    ('reg', Ridge(alpha=1.0)),
    # ('reg', LogisticRegression(multi_class='multinomial',)),
])

n_perm = 1000
np.random.seed(27)  # for reproducibility

perm_results = {
    'rho': [],
    'tau': [],
    'mae': [],
    'kappa': [],
    'r2': [],
    'rmse': []
}

for i in range(n_perm):
    print(f'Permutation {i + 1} / {n_perm}')
    y_perm_train = y_train.copy()
    np.random.shuffle(y_perm_train)

    # take random n-samples of values 1 to 9, with free duplication
    # y_perm_train = np.random.choice(
    #     np.arange(1, 10), size=y_train.shape[0], replace=True,
    # )

    # train regression model with permuted training labels
    perm_pipe.fit(X_train, y_perm_train)
    # predict test set (incl scaling of fts based on train set)
    y_pred_perm = perm_pipe.predict(X_test)
    # clip predictions to valid range and round to nearest integer
    y_pred_perm = np.clip(np.round(y_pred_perm), y_train.min(), y_train.max()).astype(int)

    # calculate metrics for permuted labels
    perm_results['rho'].append(spearmanr(y_test, y_pred_perm)[0])
    perm_results['tau'].append(kendalltau(y_test, y_pred_perm)[0])
    perm_results['mae'].append(mean_absolute_error(y_test, y_pred_perm))
    perm_results['kappa'].append(cohen_kappa_score(y_test, y_pred_perm, weights='linear'))
    perm_results['r2'].append(r2_score(y_test, y_pred_perm))
    perm_results['rmse'].append(root_mean_squared_error(y_test, y_pred_perm))
    

In [ ]:
# plot histograms of permuted metrics with observed metric as vertical line

incl_metrics = ['mae', 'rmse']

fig, axes = plt.subplots(1, len(incl_metrics), figsize=(5 * len(incl_metrics), 5))

for i, metric in enumerate(incl_metrics):

    # calculate p-value as proportion of permuted metrics that are as extreme as or more extreme than the observed metric
    if metric in ['rho', 'tau', 'kappa', 'r2']:  # for metrics where higher is better, p-value is proportion of permuted metrics >= observed metric
        p_value = np.mean(np.array(perm_results[metric]) >= test_results[metric])
    else:  # for metrics where lower is better, p-value is proportion of permuted metrics <= observed metric
        p_value = np.mean(np.array(perm_results[metric]) <= test_results[metric])
    print(f'{metric}: observed={test_results[metric]:.2f}, p-value={p_value:.4f}')

    ax = axes[i]
    ax.hist(perm_results[metric], bins=30,
            color='blue', alpha=.5, label='Permuted',)
    obs_metric = test_results[metric]  # get observed metric value from variable
    ax.axvline(obs_metric, color='red', linestyle='--', label=f'Observed {metric}={obs_metric:.2f}')
    ax.set_title(f'Permutation distribution of {metric}')
    ax.set_xlabel(metric)
    ax.set_ylabel('Frequency')
    ax.legend()
    
plt.tight_layout()
plt.show()

#### 3f) Evaluate full-day test predictions
- overall values preop vs postop
- differences in day-time-blocks
- differences around med-intakes

In [ ]:
from plotting import plot_home_preds as plt_preds

In [ ]:
SES_COLORS = {'ses01': 'violet', 'ses02': 'orange', 'ses03': 'darkcyan'}
SES_LABELS = {'ses01': 'Pre-DBS',
              'ses02': 'Post-DBS',
              'ses03': 'Post-DBS\n(optimized)'}
FS = 14

In [ ]:
# get unique ids in test data
avail_sess = np.unique(test_sessions)

for ses_A, ses_B in product(avail_sess, repeat=2):
    if ses_A >= ses_B:  # only compare each pair once and skip self-comparison
        continue
    group_A = y_test_pred[test_sessions == ses_A]
    group_B = y_test_pred[test_sessions == ses_B]
    
    # perform Mann-Whitney U test
    stat, p_value = mannwhitneyu(group_A, group_B, alternative='two-sided')
    
    print(f'Mann-Whitney U test between {ses_A} and {ses_B}: U={stat}, p={p_value:.4f}')



In [ ]:
### plot predicted LID distributions per session

fig, ax = plt.subplots(1, 1, figsize=(4, 6))

ses_groups = [y_test_pred[test_sessions == ses_id] for ses_id in avail_sess]

bp = ax.boxplot(ses_groups,
    labels=[SES_LABELS[s] for s in avail_sess],
    patch_artist=True,
    medianprops=dict(color='k')
)
for box, ses in zip(bp['boxes'], avail_sess):
    box.set_facecolor(SES_COLORS[ses])

ax.tick_params(axis='both', size=FS, labelsize=FS,)
ax.set_ylabel('Predicted Dyskinesia (EMA score)', size=FS,)
ax.set_ylim(1, 9)
plt.tight_layout()
plt.show()


In [ ]:
lid_preds_figpath = os.path.join(load_utils.get_onedrive_path('figures'), 'lid_usecase_preds')

In [ ]:
importlib.reload(data_handling)
importlib.reload(plt_preds)

save_plot = False
figname = f'fullSession_preds_LID_{FT_TYPE}_exclHR-{EXCL_HR}_logSkew-{LOG_SKEWED}_ftV-{FT_PARAMS_VERSION}.png'

fig, axes = plt.subplots(1, 2, figsize=(18, 6),
                         width_ratios=[.2, 1])

axes[0] = plt_preds.box_fullsession_preds(
    y_preds=y_test_pred, session_code=test_sessions,
    target_EMA_name='LID', ax=axes[0],
    VIOLIN=True, sign_asterix=True,
)

# axes[1] = plt_preds.plot_daily_ft_mean(
#     y_preds=y_pred, y_times=,
#     session_code=session_code, ax=axes[1],
#     ema_target_name='LID',
# )

plt.tight_layout()

if save_plot:
    plt.savefig(os.path.join(lid_preds_figpath, figname), dpi=300)
    print(f'plot saved as {figname} in {lid_preds_figpath}')
    plt.close()

else:
    plt.show()


#### Plot predictions versus l-dopa intake moments


In [ ]:
### subplots per ses

FS=14

fig, axes = plt.subplots(len(test_pred_dict), 1,
                         figsize=(3*len(test_pred_dict), 8),)

for i_ses, ses_id in enumerate(test_pred_dict):

   test_stamps = test_stamps_dict[ses_id]
   test_pred = test_pred_dict[ses_id]


   pred_dopa_dist = data_handling.get_intervals_to_ldopa(test_stamps, sub=sub_id, ses=ses_id,)
   (box_distance_groups,
    dist_labels) = data_handling.sort_values_into_ldopa_distances(
      values=test_pred, dopa_distances=pred_dopa_dist,
   )


   bp = axes[i_ses].boxplot(box_distance_groups, patch_artist=True,)
   for patch in bp['boxes']:
      patch.set_facecolor(SES_COLORS[ses_id])
      patch.set_alpha(.4)

   axes[i_ses].set_xlabel('Time vs L-Dopa Intake (minutes)', fontsize=FS,)
   axes[i_ses].set_xticklabels(dist_labels,)
   axes[i_ses].set_ylabel('Dyskinesia prediction\n(EMA scale)', fontsize=FS,)
   axes[i_ses].set_ylim(-2, 9)
   axes[i_ses].set_yticks([1, 3, 5, 7, 9])
   axes[i_ses].set_yticklabels([1, 3, 5, 7, 9])
   axes[i_ses].tick_params(size=FS, labelsize=FS,)

   axes[i_ses].set_title(ses_id, fontsize=FS,)

plt.tight_layout()

figname = 'lidPred_hm24_allSess_ldopaIntakeTime'
# plt.savefig(os.path.join(ntrl_fig_path, 'proof_kin_pred', figname),
#             facecolor='w', dpi=300,)

plt.close()

In [ ]:
### all sessions next to each other

FS=14
pos_adj = [0., .25, .5]

fig, ax = plt.subplots(1, 1, figsize=(9, 3),)

for i_ses, ses_id in enumerate(test_pred_dict):

   test_stamps = test_stamps_dict[ses_id]
   test_pred = test_pred_dict[ses_id]


   pred_dopa_dist = data_handling.get_intervals_to_ldopa(test_stamps, sub=sub_id, ses=ses_id,)
   (box_distance_groups,
    dist_labels) = data_handling.sort_values_into_ldopa_distances(
      values=test_pred, dopa_distances=pred_dopa_dist,
   )

   bp = ax.boxplot(
      box_distance_groups, patch_artist=True, label=ses_id,
      positions=np.arange(len(box_distance_groups)) + pos_adj[i_ses],
      widths=.25,
   )

   for patch in bp['boxes']:
      patch.set_facecolor(SES_COLORS[ses_id])
      patch.set_alpha(.4)

ax.set_xlabel('Time vs L-Dopa Intake (minutes)', fontsize=FS,)
ax.set_xticks(np.arange(len(dist_labels))+pos_adj[1],)
ax.set_xticklabels(dist_labels,)
ax.set_ylabel('Dyskinesia prediction\n(EMA scale)', fontsize=FS,)
ax.set_ylim(-3, 9)
ax.set_yticks([-3, -1, 1, 3, 5, 7, 9])
ax.set_yticklabels([-3, -1, 1, 3, 5, 7, 9])
ax.tick_params(size=FS, labelsize=FS,)

ax.legend(ncol=3, frameon=False, fontsize=FS,
          loc='lower center', bbox_to_anchor=(.5, 1))

plt.tight_layout()

figname = 'lidPred_hm24_allSess_ldopaIntakeTime'
plt.savefig(os.path.join(ntrl_fig_path, 'proof_kin_pred', figname),
            facecolor='w', dpi=300,)

plt.close()

Daytime blocks

In [ ]:
DAY_HOUR_BLOCKS = {'morning': [8, 12],
                   'afternoon1': [12, 15],
                   'afternoon2': [15, 18], 
                   'evening': [18, 22]}

box_lists = {k: {k2: [] for k2 in DAY_HOUR_BLOCKS}
             for k in test_pred_dict}


for ses_id in test_pred_dict.keys():

   test_stamps = test_stamps_dict[ses_id]
   test_pred = test_pred_dict[ses_id]
   stamp_hrs = [data_handling.get_dayminutes(t)/60 for t in test_stamps]
   
   for t_hr, v in zip(stamp_hrs, test_pred):
      if t_hr < DAY_HOUR_BLOCKS['morning'][1]:
         box_lists[ses_id]['morning'].append(v)
      elif t_hr < DAY_HOUR_BLOCKS['afternoon1'][1]:
         box_lists[ses_id]['afternoon1'].append(v)
      elif t_hr < DAY_HOUR_BLOCKS['afternoon2'][1]:
         box_lists[ses_id]['afternoon2'].append(v)
      else:
         box_lists[ses_id]['evening'].append(v)

boxparams = {
   'ses01': {'widths': .2, 'positions': np.arange(.0, len(DAY_HOUR_BLOCKS)),},
   'ses02': {'widths': .2, 'positions': np.arange(.2, len(DAY_HOUR_BLOCKS)),},
   'ses03': {'widths': .2, 'positions': np.arange(.4, len(DAY_HOUR_BLOCKS)),}
}
FS=14

fig, ax = plt.subplots(1, 1, figsize=(8, 4))

for i_ses, ses_id in enumerate(box_lists):
   ses_lists = [l for l in box_lists[ses_id].values()]
   bp = ax.boxplot(ses_lists, **boxparams[ses_id],
                   patch_artist=True, label=ses_id)
   for patch in bp['boxes']:
      patch.set_facecolor(SES_COLORS[ses_id])
      patch.set_alpha(.4)

ax.legend(ncol=3, frameon=False, fontsize=FS,
          loc='lower center', bbox_to_anchor=(.5, 1))
ax.set_xticks(list(boxparams.values())[1]['positions'])
ax.set_xticklabels(list(DAY_HOUR_BLOCKS.keys()))
ax.set_ylabel('Dyskinesia prediction\n(EMA scale)', fontsize=FS,)
ax.tick_params(labelsize=FS, size=FS,)

plt.tight_layout()

figname = 'lidPred_hm24_allSess_daytimeBlocks'
plt.savefig(os.path.join(ntrl_fig_path, 'proof_kin_pred', figname),
            facecolor='w', dpi=300,)

plt.close()